In [3]:
import time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
import math
import warnings

warnings.filterwarnings('ignore')

GLOBAL_PARAMS = {
    'TEST_SIZE': 0.20,
    'RANDOM_STATE': 42,
    'K_USERS': 30,
    'K_ITEMS': 20,
    'TOP_X': 10,
    'MIN_RATING_LIKE': 4,
    'ALPHA_UB': 0.55,
    'INTERSECTION_BONUS': 1.25
}

DATA_DIR = 'goodbooks-10k-extended'


def load_and_inspect_data():
    ratings = pd.read_csv(
        f'{DATA_DIR}/ratings.csv',
        usecols=['user_id', 'book_id', 'rating']
    )

    books = pd.read_csv(
        f'{DATA_DIR}/books_enriched.csv',
        index_col=0
    )
    books.index.name = 'book_id'

    books['book_id'] = pd.to_numeric(books['book_id'], errors='coerce')
    ratings['user_id'] = pd.to_numeric(ratings['user_id'], errors='coerce')
    ratings['book_id'] = pd.to_numeric(ratings['book_id'], errors='coerce')
    ratings['rating'] = pd.to_numeric(ratings['rating'], errors='coerce')

    ratings = ratings.dropna(subset=['user_id', 'book_id', 'rating'])
    books = books.dropna(subset=['book_id'])

    ratings['user_id'] = ratings['user_id'].astype(np.int32)
    ratings['book_id'] = ratings['book_id'].astype(np.int32)
    books['book_id'] = books['book_id'].astype(np.int32)

    n_users = ratings['user_id'].nunique()
    n_books = ratings['book_id'].nunique()
    n_ratings = len(ratings)

    total_possible = n_users * n_books
    sparsity = (1.0 - (n_ratings / total_possible)) * 100

    print("=" * 55)
    print("АНАЛИЗ СОДЕРЖАНИЯ ДАННЫХ (Goodbooks-10k Extended)")
    print("=" * 55)
    print(f"Уникальных пользователей: {n_users:,}")
    print(f"Уникальных книг:          {n_books:,}")
    print(f"Всего выставлено оценок:  {n_ratings:,}")
    print(f"Разреженность матрицы:    {sparsity:.2f}%")
    print(f"Средний рейтинг по базе:  {ratings['rating'].mean():.2f} / 5.0")
    print("=" * 55)

    return ratings, books


ratings_df, books_df = load_and_inspect_data()

books_meta = books_df.set_index('book_id')[['title', 'genres']]




АНАЛИЗ СОДЕРЖАНИЯ ДАННЫХ (Goodbooks-10k Extended)
Уникальных пользователей: 53,424
Уникальных книг:          10,000
Всего выставлено оценок:  5,976,479
Разреженность матрицы:    98.88%
Средний рейтинг по базе:  3.92 / 5.0


In [4]:
def prepare_matrices(ratings, test_size=GLOBAL_PARAMS['TEST_SIZE']):
    train_df, test_df = train_test_split(
        ratings,
        test_size=test_size,
        random_state=GLOBAL_PARAMS['RANDOM_STATE'],
        stratify=None
    )

    user_ids = pd.Index(np.sort(train_df['user_id'].unique()))
    item_ids = pd.Index(np.sort(train_df['book_id'].unique()))

    user_to_idx = pd.Series(
        np.arange(len(user_ids)),
        index=user_ids
    )

    item_to_idx = pd.Series(
        np.arange(len(item_ids)),
        index=item_ids
    )

    row_idx = train_df['user_id'].map(user_to_idx).to_numpy()
    col_idx = train_df['book_id'].map(item_to_idx).to_numpy()
    values = train_df['rating'].astype(np.float32).to_numpy()

    ui_sparse = csr_matrix(
        (values, (row_idx, col_idx)),
        shape=(len(user_ids), len(item_ids)),
        dtype=np.float32
    )

    ui_sparse.sum_duplicates()
    iu_sparse = ui_sparse.T.tocsr()

    model_ub = NearestNeighbors(
        metric='cosine',
        algorithm='brute',
        n_neighbors=min(GLOBAL_PARAMS['K_USERS'] + 1, len(user_ids)),
        n_jobs=-1
    )
    model_ub.fit(ui_sparse)

    model_ib = NearestNeighbors(
        metric='cosine',
        algorithm='brute',
        n_neighbors=min(GLOBAL_PARAMS['K_ITEMS'] + 1, len(item_ids)),
        n_jobs=-1
    )
    model_ib.fit(iu_sparse)

    return (
        model_ub,
        model_ib,
        ui_sparse,
        iu_sparse,
        user_ids,
        item_ids,
        user_to_idx,
        item_to_idx,
        train_df,
        test_df
    )


(
    model_ub,
    model_ib,
    ui_matrix,
    iu_matrix,
    user_ids,
    item_ids,
    user_to_idx,
    item_to_idx,
    train_df,
    test_df
) = prepare_matrices(ratings_df)

In [5]:
def get_hybrid_recommendations(
    user_id,
    model_ub,
    model_ib,
    ui_matrix,
    iu_matrix,
    user_ids,
    item_ids,
    user_to_idx,
    item_to_idx,
    books_meta,
    top_x=GLOBAL_PARAMS['TOP_X'],
    alpha=GLOBAL_PARAMS['ALPHA_UB'],
    bonus=GLOBAL_PARAMS['INTERSECTION_BONUS'],
    min_support=2
):
    if user_id not in user_to_idx.index:
        return pd.DataFrame()

    user_idx = int(user_to_idx.loc[user_id])
    user_vec = ui_matrix.getrow(user_idx)
    user_ratings = user_vec.toarray().ravel()

    unseen_mask = user_ratings == 0
    unseen_item_positions = np.flatnonzero(unseen_mask)
    unseen_book_ids = item_ids[unseen_item_positions]

    n_neighbors_ub = min(
        GLOBAL_PARAMS['K_USERS'] + 1,
        ui_matrix.shape[0]
    )

    dist_u, ind_u = model_ub.kneighbors(
        user_vec,
        n_neighbors=n_neighbors_ub
    )

    neighbor_u_indices = ind_u.flatten()[1:]
    sim_users = 1.0 - dist_u.flatten()[1:]

    if len(unseen_item_positions) > 0 and len(neighbor_u_indices) > 0:
        neighbor_ratings = ui_matrix[
            neighbor_u_indices
        ][:, unseen_item_positions].toarray()

        sim_weights = sim_users[:, np.newaxis]

        num_neighbors_rated = np.sum(
            neighbor_ratings > 0,
            axis=0
        )

        weighted_sum = np.sum(
            neighbor_ratings * sim_weights,
            axis=0
        )

        sum_sims = np.sum(
            sim_weights * (neighbor_ratings > 0),
            axis=0
        )

        sum_sims[sum_sims == 0] = 1e-9

        ub_predicted = weighted_sum / sum_sims

        valid_ub_mask = (
            (ub_predicted > 0) &
            (num_neighbors_rated >= min_support)
        )

        ub_candidates = pd.Series(
            ub_predicted[valid_ub_mask],
            index=unseen_book_ids[valid_ub_mask]
        )
    else:
        ub_candidates = pd.Series(dtype=float)

    liked_positions = np.flatnonzero(
        user_ratings >= GLOBAL_PARAMS['MIN_RATING_LIKE']
    )

    ib_scores = {}

    n_neighbors_ib = min(
        GLOBAL_PARAMS['K_ITEMS'] + 1,
        iu_matrix.shape[0]
    )

    unseen_set = set(unseen_book_ids)

    for m_pos in liked_positions:
        m_id = item_ids[m_pos]

        if m_id not in item_to_idx.index:
            continue

        m_idx = int(item_to_idx.loc[m_id])
        item_vec = iu_matrix.getrow(m_idx)

        dist_i, ind_i = model_ib.kneighbors(
            item_vec,
            n_neighbors=n_neighbors_ib
        )

        for c_idx, c_dist in zip(
            ind_i.flatten()[1:],
            dist_i.flatten()[1:]
        ):
            candidate_id = item_ids[c_idx]

            if candidate_id not in unseen_set:
                continue

            similarity = 1.0 - c_dist
            weight = user_ratings[m_pos] / 5.0

            ib_scores[candidate_id] = (
                ib_scores.get(candidate_id, 0.0)
                + similarity * weight
            )

    ib_candidates = pd.Series(ib_scores, dtype=float)

    all_candidates = list(
        set(ub_candidates.index) |
        set(ib_candidates.index)
    )

    if not all_candidates:
        return pd.DataFrame()

    df_fusion = pd.DataFrame(index=all_candidates)
    df_fusion['ub_score'] = ub_candidates
    df_fusion['ib_score'] = ib_candidates
    df_fusion = df_fusion.fillna(0.0)

    def min_max_scale(series):
        if series.empty:
            return pd.Series(dtype=float)

        denom = series.max() - series.min()

        if denom > 0:
            return (series - series.min()) / denom

        return pd.Series(
            0.0,
            index=series.index
        )

    if not ub_candidates.empty:
        df_fusion['ub_norm'] = min_max_scale(
            ub_candidates
        ).reindex(df_fusion.index).fillna(0.0)
    else:
        df_fusion['ub_norm'] = 0.0

    if not ib_candidates.empty:
        df_fusion['ib_norm'] = min_max_scale(
            ib_candidates
        ).reindex(df_fusion.index).fillna(0.0)
    else:
        df_fusion['ib_norm'] = 0.0

    df_fusion['hybrid_score'] = (
        alpha * df_fusion['ub_norm']
        + (1.0 - alpha) * df_fusion['ib_norm']
    )

    has_ub = df_fusion['ub_score'] > 0
    has_ib = df_fusion['ib_score'] > 0
    in_both_mask = has_ub & has_ib

    df_fusion['source'] = 'Unique'
    df_fusion.loc[
        has_ub & ~has_ib,
        'source'
    ] = 'UB-only'
    df_fusion.loc[
        has_ib & ~has_ub,
        'source'
    ] = 'IB-only'
    df_fusion.loc[
        in_both_mask,
        'source'
    ] = 'INTERSECTION [UB+IB]'

    df_fusion.loc[
        in_both_mask,
        'hybrid_score'
    ] *= bonus

    ranked = df_fusion.sort_values(
        by='hybrid_score',
        ascending=False
    ).head(top_x)

    ranked = ranked.join(
        books_meta,
        how='left'
    )

    return ranked[
        [
            'title',
            'genres',
            'source',
            'hybrid_score',
            'ub_score',
            'ib_score'
        ]
    ]

In [6]:
def compute_ranking_metrics_for_user(
    user_id,
    recommended_ids,
    test_df,
    k=10,
    relevance_threshold=4
):
    user_test = test_df[
        test_df['user_id'] == user_id
    ]

    actual_relevant = set(
        user_test[
            user_test['rating'] >= relevance_threshold
        ]['book_id']
    )

    if not actual_relevant:
        return None

    hits = [
        1 if book_id in actual_relevant else 0
        for book_id in recommended_ids[:k]
    ]

    num_hits = sum(hits)

    precision = num_hits / float(k)
    recall = num_hits / float(len(actual_relevant))
    hit_rate = 1.0 if num_hits > 0 else 0.0

    dcg = sum(
        hit / math.log2(idx + 2)
        for idx, hit in enumerate(hits)
    )

    ideal_hits = [
        1
    ] * min(k, len(actual_relevant))

    idcg = sum(
        1.0 / math.log2(idx + 2)
        for idx in range(len(ideal_hits))
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {
        'precision': precision,
        'recall': recall,
        'hit_rate': hit_rate,
        'ndcg': ndcg
    }


def evaluate_system(
    eval_user_ids,
    model_ub,
    model_ib,
    ui_matrix,
    iu_matrix,
    user_ids,
    item_ids,
    user_to_idx,
    item_to_idx,
    test_df,
    k=10,
    **params
):
    metrics_list = []

    for uid in eval_user_ids:
        recs_df = get_hybrid_recommendations(
            user_id=uid,
            model_ub=model_ub,
            model_ib=model_ib,
            ui_matrix=ui_matrix,
            iu_matrix=iu_matrix,
            user_ids=user_ids,
            item_ids=item_ids,
            user_to_idx=user_to_idx,
            item_to_idx=item_to_idx,
            books_meta=books_meta,
            top_x=k,
            **params
        )

        if recs_df.empty:
            continue

        metrics = compute_ranking_metrics_for_user(
            uid,
            recs_df.index.tolist(),
            test_df,
            k=k
        )

        if metrics:
            metrics_list.append(metrics)

    if not metrics_list:
        return {
            'Precision@K': 0,
            'Recall@K': 0,
            'HitRate@K': 0,
            'NDCG@K': 0
        }

    results = pd.DataFrame(
        metrics_list
    ).mean().to_dict()

    return {
        'Precision@K': results['precision'],
        'Recall@K': results['recall'],
        'HitRate@K': results['hit_rate'],
        'NDCG@K': results['ndcg']
    }

In [8]:
np.random.seed(42)
available_eval_users = train_df['user_id'].unique()
sample_eval_users = np.random.choice(available_eval_users, size=min(200, len(available_eval_users)), replace=False)

print(f"Шаг 1: Предрасчет кандидатов для {len(sample_eval_users)} пользователей...")
start_time = time.time()
cached_user_data = {}

for i, uid in enumerate(sample_eval_users, 1):
    if uid not in user_to_idx.index: continue

    user_idx = int(user_to_idx.loc[uid])
    user_vec = ui_matrix.getrow(user_idx)
    user_ratings = user_vec.toarray().ravel()
    unseen_positions = np.flatnonzero(user_ratings == 0)
    unseen_ids = item_ids[unseen_positions]
    unseen_set = set(unseen_ids)

    n_neighbors_ub = min(GLOBAL_PARAMS['K_USERS'] + 1, ui_matrix.shape[0])
    dist_u, ind_u = model_ub.kneighbors(user_vec, n_neighbors=n_neighbors_ub)
    neighbor_indices = ind_u.flatten()[1:]
    sim_u = 1.0 - dist_u.flatten()[1:]

    if len(unseen_positions) > 0:
        n_ratings = ui_matrix[neighbor_indices][:, unseen_positions].toarray()
        sim_weights = sim_u[:, np.newaxis]
        num_rated = np.sum(n_ratings > 0, axis=0)
        weighted_sum = np.sum(n_ratings * sim_weights, axis=0)
        sum_sim = np.sum(sim_weights * (n_ratings > 0), axis=0)
        sum_sim[sum_sim == 0] = 1e-9
        ub_pred = weighted_sum / sum_sim
        mask = (ub_pred > 0) & (num_rated >= 2)
        ub_cand = pd.Series(ub_pred[mask], index=unseen_ids[mask])
    else:
        ub_cand = pd.Series(dtype=float)

    liked_positions = np.flatnonzero(user_ratings >= GLOBAL_PARAMS['MIN_RATING_LIKE'])
    if len(liked_positions) > 20: liked_positions = liked_positions[-20:]

    ib_dict = {}
    n_neighbors_ib = min(GLOBAL_PARAMS['K_ITEMS'] + 1, iu_matrix.shape[0])

    for m_pos in liked_positions:
        m_id = item_ids[m_pos]
        if m_id not in item_to_idx.index: continue
        m_idx = int(item_to_idx.loc[m_id])
        item_vec = iu_matrix.getrow(m_idx)
        dist_i, ind_i = model_ib.kneighbors(item_vec, n_neighbors=n_neighbors_ib)

        for c_idx, c_dist in zip(ind_i.flatten()[1:], dist_i.flatten()[1:]):
            candidate_id = item_ids[c_idx]
            if candidate_id not in unseen_set: continue
            ib_dict[candidate_id] = ib_dict.get(candidate_id, 0.0) + (1.0 - c_dist) * (user_ratings[m_pos] / 5.0)

    ib_cand = pd.Series(ib_dict, dtype=float)

    def norm_s(series):
        if series.empty: return pd.Series(dtype=float)
        d = series.max() - series.min()
        return (series - series.min()) / d if d > 0 else pd.Series(0.0, index=series.index)

    all_cands = list(set(ub_cand.index) | set(ib_cand.index))
    if all_cands:
        df_c = pd.DataFrame(index=all_cands)
        df_c['ub'] = norm_s(ub_cand).reindex(df_c.index).fillna(0.0) if not ub_cand.empty else 0.0
        df_c['ib'] = norm_s(ib_cand).reindex(df_c.index).fillna(0.0) if not ib_cand.empty else 0.0
        cached_user_data[uid] = df_c

    if i == 1 or i % 10 == 0 or i == len(sample_eval_users):
        elapsed = time.time() - start_time
        speed = i / elapsed if elapsed > 0 else 0
        eta = (len(sample_eval_users) - i) / speed if speed > 0 else 0
        print(f"Обработано: {i}/{len(sample_eval_users)} ({i / len(sample_eval_users) * 100:.1f}%), скорость: {speed:.1f} польз./с, осталось: {eta:.1f} сек.")

print(f"Кэширование завершено за {time.time() - start_time:.1f} сек.\n")


def evaluate_cached_grid(cached_data, test_df, alpha=0.5, bonus=1.2, top_x=10):
    metrics = []
    total = len(cached_data)

    for i, (uid, df_c) in enumerate(cached_data.items(), 1):
        scores = alpha * df_c['ub'] + (1.0 - alpha) * df_c['ib']
        in_both = (df_c['ub'] > 0) & (df_c['ib'] > 0)
        scores.loc[in_both] *= bonus
        top_recs = scores.nlargest(top_x).index.tolist()
        metrics_user = compute_ranking_metrics_for_user(uid, top_recs, test_df, k=top_x)
        if metrics_user: metrics.append(metrics_user)

        if i == 1 or i % 25 == 0 or i == total:
            print(f"  Оценка: {i}/{total} ({i / total * 100:.1f}%)", end='\r')

    print()
    if not metrics: return {'Precision@K': 0, 'Recall@K': 0, 'NDCG@K': 0, 'HitRate@K': 0}

    res = pd.DataFrame(metrics).mean().to_dict()
    return {'Precision@K': res['precision'], 'Recall@K': res['recall'], 'NDCG@K': res['ndcg'], 'HitRate@K': res['hit_rate']}


print("Эксперимент 1: Исследование веса Alpha...")
alpha_res = []

for j, a in enumerate([0.0, 0.1, 0.2, 0.35, 0.5, 0.7, 1.0], 1):
    print(f"  Alpha {j}/7: {a}")
    r = evaluate_cached_grid(cached_user_data, test_df, alpha=a, bonus=1.0, top_x=GLOBAL_PARAMS['TOP_X'])
    r['Alpha (UB Weight)'] = a
    alpha_res.append(r)

df_alpha = pd.DataFrame(alpha_res).set_index('Alpha (UB Weight)')
display(df_alpha[['Precision@K', 'Recall@K', 'NDCG@K', 'HitRate@K']])
best_alpha = df_alpha['NDCG@K'].idxmax()
print(f"-> Лучший Alpha по NDCG: {best_alpha}")


print("\nЭксперимент 2: Исследование бонуса за пересечение...")
bonus_res = []

for j, b in enumerate([1.0, 1.1, 1.25, 1.5, 2.0], 1):
    print(f"  Bonus {j}/5: {b}")
    r = evaluate_cached_grid(cached_user_data, test_df, alpha=best_alpha, bonus=b, top_x=GLOBAL_PARAMS['TOP_X'])
    r['Bonus Multiplier'] = b
    bonus_res.append(r)

df_bonus = pd.DataFrame(bonus_res).set_index('Bonus Multiplier')
display(df_bonus[['Precision@K', 'Recall@K', 'NDCG@K', 'HitRate@K']])


print("\nЭксперимент 3: Влияние длины выдачи Top-X...")
top_res = []

for j, x in enumerate([5, 10, 15, 20], 1):
    print(f"  Top-X {j}/4: {x}")
    r = evaluate_cached_grid(cached_user_data, test_df, alpha=best_alpha, bonus=1.2, top_x=x)
    r['Top-X'] = x
    top_res.append(r)

df_topx = pd.DataFrame(top_res).set_index('Top-X')
display(df_topx[['Precision@K', 'Recall@K', 'NDCG@K', 'HitRate@K']])

Шаг 1: Предрасчет кандидатов для 200 пользователей...
Обработано: 1/200 (0.5%), скорость: 0.1 польз./с, осталось: 1806.3 сек.
Обработано: 10/200 (5.0%), скорость: 0.2 польз./с, осталось: 822.4 сек.
Обработано: 20/200 (10.0%), скорость: 0.2 польз./с, осталось: 751.8 сек.
Обработано: 30/200 (15.0%), скорость: 0.2 польз./с, осталось: 697.6 сек.
Обработано: 40/200 (20.0%), скорость: 0.2 польз./с, осталось: 651.4 сек.
Обработано: 50/200 (25.0%), скорость: 0.2 польз./с, осталось: 606.7 сек.
Обработано: 60/200 (30.0%), скорость: 0.2 польз./с, осталось: 563.7 сек.
Обработано: 70/200 (35.0%), скорость: 0.2 польз./с, осталось: 521.1 сек.
Обработано: 80/200 (40.0%), скорость: 0.3 польз./с, осталось: 474.4 сек.
Обработано: 90/200 (45.0%), скорость: 0.3 польз./с, осталось: 431.3 сек.
Обработано: 100/200 (50.0%), скорость: 0.3 польз./с, осталось: 391.7 сек.
Обработано: 110/200 (55.0%), скорость: 0.3 польз./с, осталось: 352.6 сек.
Обработано: 120/200 (60.0%), скорость: 0.3 польз./с, осталось: 313.3 с

,Precision@K,Recall@K,NDCG@K,HitRate@K
Alpha (UB Weight),,,,
0.00,0.198485,0.135988,0.250201,0.808081
0.10,0.204545,0.139421,0.255959,0.818182
0.20,0.210606,0.144350,0.259665,0.833333
0.35,0.204545,0.141195,0.253105,0.838384
0.50,0.181313,0.124880,0.228254,0.792929
0.70,0.133333,0.091832,0.172716,0.686869
1.00,0.012121,0.008750,0.010960,0.121212


-> Лучший Alpha по NDCG: 0.2

Эксперимент 2: Исследование бонуса за пересечение...
  Bonus 1/5: 1.0
  Оценка: 200/200 (100.0%)
  Bonus 2/5: 1.1
  Оценка: 200/200 (100.0%)
  Bonus 3/5: 1.25
  Оценка: 200/200 (100.0%)
  Bonus 4/5: 1.5
  Оценка: 200/200 (100.0%)
  Bonus 5/5: 2.0
  Оценка: 200/200 (100.0%)


,Precision@K,Recall@K,NDCG@K,HitRate@K
Bonus Multiplier,,,,
1.00,0.210606,0.144350,0.259665,0.833333
1.10,0.210606,0.143804,0.257892,0.833333
1.25,0.209596,0.142324,0.257092,0.838384
1.50,0.214646,0.144707,0.261220,0.833333
2.00,0.215657,0.145975,0.261543,0.833333



Эксперимент 3: Влияние длины выдачи Top-X...
  Top-X 1/4: 5
  Оценка: 200/200 (100.0%)
  Top-X 2/4: 10
  Оценка: 200/200 (100.0%)
  Top-X 3/4: 15
  Оценка: 200/200 (100.0%)
  Top-X 4/4: 20
  Оценка: 200/200 (100.0%)


,Precision@K,Recall@K,NDCG@K,HitRate@K
Top-X,,,,
5,0.259596,0.089313,0.296754,0.696970
10,0.209596,0.142829,0.257220,0.838384
15,0.177441,0.184303,0.242638,0.868687
20,0.156566,0.212474,0.241404,0.883838
